### Transform Constructors Table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F

####Step 1 - Read the data from the bronze table

In [0]:
constructors_df = spark.table(bronze_table)

####Step 2 - Drop URL column as it is not needed

In [0]:
constructors_dropped_df = constructors_df.drop("url")

####Step 3 & 4 - Rename columns in snake case and readable

In [0]:
constructors_renamed_df = constructors_dropped_df.withColumnsRenamed({
    "constructorId" : "constructor_id",
    "name" : "constructor_name"
})

####Step 5 - Remove Duplicate Records

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

####Step 6 - Transform values of column nationality to Title Case

In [0]:
constructors_final_df = constructors_distinct_df.withColumn("nationality", F.initcap(F.col("nationality")))

####Step 7 - Write the data into silver constructors table

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)